# EDA: is TMDB `cast_order` a reliable billing rank?

Run this **before** implementing `src/features/star_power.py`. This notebook only decides a boolean; it does not implement star-power math.

Kleos has two formulas, switched by `config.USE_WEIGHTED_STAR_POWER`:

- **Path A** (`True`): inverse-`cast_order` weights over the **full** cast. Requires `order` to be real billing, not alphabetical or junk.
- **Path B** (`False`): unweighted mean of the top 5 names by `cast_order`. Safer if `order` is noisy.

Data: `data/processed/movies_with_credits.parquet` (financially filtered movies + `credits_cache.jsonl`). Cast members are `{id, name, order}`.

TMDB `order` is **0-based** (first-billed = `0`). Path A must use `1 / (order + 1)` at implement time so the lead is not a divide-by-zero.

## Sample design

Twenty titles picked **by name/id**, not a random draw — we need films whose real leads we know. Mix:

| Category | Titles |
|---|---|
| Ensemble blockbuster | The Avengers (2012), Dune (2021), Oppenheimer (2023) |
| Big film, clear leads | The Dark Knight (2008), Inception (2010), Avatar (2009), Barbie (2023) |
| Two- or three-hander | Titanic (1997), La La Land (2016), The Social Network (2010), Get Out (2017) |
| Indie / small cast | Moonlight (2016), Everything Everywhere All at Once (2022) |
| Pre-2000 | The Godfather (1972), Jaws (1975), Pulp Fiction (1994), The Shawshank Redemption (1994), Titanic (1997) |
| Non-English / animation | Parasite (2019), Spirited Away (2001) |
| Billed-first ≠ folk "protagonist" | The Godfather, Mad Max: Fury Road, Parasite, Moonlight |

A random sample of the 900k dump would not have ground truth. This sample is biased toward well-known, well-curated TMDB pages — the same bias as our financially filtered set (budget > 0 and revenue > 0), which is what star power will actually see.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import MOVIES_WITH_CREDITS_PATH, USE_WEIGHTED_STAR_POWER

pd.set_option("display.max_colwidth", 120)


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).casefold())


def name_matches(candidate: str, known: list[str]) -> bool:
    cand = normalize_name(candidate)
    if not cand:
        return False
    for lead in known:
        lead_n = normalize_name(lead)
        if cand == lead_n or cand in lead_n or lead_n in cand:
            return True
    return False


def sort_cast(cast) -> list[dict]:
    if not isinstance(cast, list):
        return []
    return sorted(
        (m for m in cast if isinstance(m, dict)),
        key=lambda m: (m.get("order") is None, m.get("order") if m.get("order") is not None else 10**9),
    )


def looks_alphabetical(names: list[str]) -> bool:
    if len(names) < 5:
        return False
    return [n.casefold() for n in names] == sorted((n.casefold() for n in names))


print("config.USE_WEIGHTED_STAR_POWER (current):", USE_WEIGHTED_STAR_POWER)
print("parquet:", MOVIES_WITH_CREDITS_PATH, "exists=", MOVIES_WITH_CREDITS_PATH.exists())

In [ ]:
# Ground-truth leads from film knowledge (poster / who the movie is "about"),
# not copied from TMDB. Aliases cover TMDB name-order and credit changes.
SAMPLE = [
    {
        "id": 24428,
        "title": "The Avengers",
        "year": 2012,
        "category": "ensemble blockbuster",
        "known_leads": [
            "Robert Downey Jr.", "Chris Evans", "Chris Hemsworth",
            "Scarlett Johansson", "Mark Ruffalo", "Jeremy Renner",
        ],
    },
    {
        "id": 155,
        "title": "The Dark Knight",
        "year": 2008,
        "category": "blockbuster, clear leads",
        "known_leads": ["Christian Bale", "Heath Ledger", "Aaron Eckhart"],
    },
    {
        "id": 27205,
        "title": "Inception",
        "year": 2010,
        "category": "blockbuster, clear leads",
        "known_leads": [
            "Leonardo DiCaprio", "Joseph Gordon-Levitt", "Elliot Page",
            "Ellen Page", "Tom Hardy", "Marion Cotillard",
        ],
    },
    {
        "id": 597,
        "title": "Titanic",
        "year": 1997,
        "category": "two-hander / pre-2000 blockbuster",
        "known_leads": ["Leonardo DiCaprio", "Kate Winslet", "Billy Zane"],
    },
    {
        "id": 19995,
        "title": "Avatar",
        "year": 2009,
        "category": "blockbuster, clear leads",
        "known_leads": ["Sam Worthington", "Zoe Saldana", "Zoe Saldaña", "Sigourney Weaver"],
    },
    {
        "id": 238,
        "title": "The Godfather",
        "year": 1972,
        "category": "pre-2000 / billed ≠ sole protagonist",
        "known_leads": ["Marlon Brando", "Al Pacino", "James Caan"],
    },
    {
        "id": 680,
        "title": "Pulp Fiction",
        "year": 1994,
        "category": "pre-2000 / special billing",
        "known_leads": ["John Travolta", "Samuel L. Jackson", "Uma Thurman", "Bruce Willis"],
    },
    {
        "id": 419430,
        "title": "Get Out",
        "year": 2017,
        "category": "mid-budget two-hander",
        "known_leads": ["Daniel Kaluuya", "Allison Williams", "Catherine Keener"],
    },
    {
        "id": 496243,
        "title": "Parasite",
        "year": 2019,
        "category": "non-English",
        "known_leads": [
            "Song Kang-ho", "Kang-ho Song", "Lee Sun-kyun", "Sun-kyun Lee",
            "Choi Woo-shik", "Woo-shik Choi", "Cho Yeo-jeong", "Park So-dam",
        ],
    },
    {
        "id": 76341,
        "title": "Mad Max: Fury Road",
        "year": 2015,
        "category": "billed ≠ perceived lead",
        "known_leads": ["Tom Hardy", "Charlize Theron", "Nicholas Hoult"],
    },
    {
        "id": 313369,
        "title": "La La Land",
        "year": 2016,
        "category": "two-hander",
        "known_leads": ["Ryan Gosling", "Emma Stone"],
    },
    {
        "id": 37799,
        "title": "The Social Network",
        "year": 2010,
        "category": "mid-budget two/three-hander",
        "known_leads": ["Jesse Eisenberg", "Andrew Garfield", "Justin Timberlake", "Armie Hammer"],
    },
    {
        "id": 346698,
        "title": "Barbie",
        "year": 2023,
        "category": "recent blockbuster",
        "known_leads": ["Margot Robbie", "Ryan Gosling", "America Ferrera"],
    },
    {
        "id": 872585,
        "title": "Oppenheimer",
        "year": 2023,
        "category": "ensemble blockbuster",
        "known_leads": ["Cillian Murphy", "Emily Blunt", "Matt Damon", "Robert Downey Jr."],
    },
    {
        "id": 545611,
        "title": "Everything Everywhere All at Once",
        "year": 2022,
        "category": "indie / small-cast family",
        "known_leads": ["Michelle Yeoh", "Ke Huy Quan", "Stephanie Hsu", "Jamie Lee Curtis"],
    },
    {
        "id": 578,
        "title": "Jaws",
        "year": 1975,
        "category": "pre-2000 three-hander",
        "known_leads": ["Roy Scheider", "Robert Shaw", "Richard Dreyfuss"],
    },
    {
        "id": 129,
        "title": "Spirited Away",
        "year": 2001,
        "category": "non-English animation",
        "known_leads": ["Rumi Hiiragi", "Miyu Irino", "Mari Natsuki"],
    },
    {
        "id": 278,
        "title": "The Shawshank Redemption",
        "year": 1994,
        "category": "pre-2000 two-hander",
        "known_leads": ["Tim Robbins", "Morgan Freeman", "Bob Gunton"],
    },
    {
        "id": 438631,
        "title": "Dune",
        "year": 2021,
        "category": "ensemble blockbuster",
        "known_leads": ["Timothée Chalamet", "Timothee Chalamet", "Rebecca Ferguson", "Zendaya", "Oscar Isaac"],
    },
    {
        "id": 376867,
        "title": "Moonlight",
        "year": 2016,
        "category": "indie / three-actor lead",
        "known_leads": [
            "Trevante Rhodes", "André Holland", "Andre Holland",
            "Ashton Sanders", "Alex Hibbert", "Alex R. Hibbert",
            "Mahershala Ali", "Naomie Harris",
        ],
    },
]

# Live TMDB /cast pages, 2026-09-10. Used only when a title is missing from the parquet
# (not yet merged, or dropped by the financial filter). Same `order` field as the API.
TMDB_TOP_BILLED_FALLBACK = {
    24428: ["Robert Downey Jr.", "Chris Evans", "Mark Ruffalo", "Chris Hemsworth", "Scarlett Johansson", "Jeremy Renner", "Tom Hiddleston", "Clark Gregg", "Cobie Smulders", "Stellan Skarsgård"],
    155: ["Christian Bale", "Heath Ledger", "Aaron Eckhart", "Michael Caine", "Maggie Gyllenhaal", "Gary Oldman", "Morgan Freeman", "Monique Gabriela Curnen", "Ron Dean", "Cillian Murphy"],
    27205: ["Leonardo DiCaprio", "Joseph Gordon-Levitt", "Ken Watanabe", "Tom Hardy", "Elliot Page", "Dileep Rao", "Cillian Murphy", "Tom Berenger", "Marion Cotillard", "Pete Postlethwaite"],
    597: ["Leonardo DiCaprio", "Kate Winslet", "Billy Zane", "Kathy Bates", "Frances Fisher", "Gloria Stuart", "Bill Paxton", "Bernard Hill", "David Warner", "Victor Garber"],
    19995: ["Sam Worthington", "Zoe Saldaña", "Sigourney Weaver", "Stephen Lang", "Michelle Rodriguez", "Giovanni Ribisi", "Joel David Moore", "CCH Pounder", "Wes Studi", "Laz Alonso"],
    238: ["Marlon Brando", "Al Pacino", "James Caan", "Robert Duvall", "Richard S. Castellano", "Diane Keaton", "Talia Shire", "Gianni Russo", "Sterling Hayden", "John Marley"],
    680: ["John Travolta", "Samuel L. Jackson", "Uma Thurman", "Bruce Willis", "Ving Rhames", "Harvey Keitel", "Eric Stoltz", "Tim Roth", "Amanda Plummer", "Maria de Medeiros"],
    419430: ["Daniel Kaluuya", "Allison Williams", "Catherine Keener", "Bradley Whitford", "Caleb Landry Jones", "Marcus Henderson", "Betty Gabriel", "LaKeith Stanfield", "Stephen Root", "Lil Rel Howery"],
    496243: ["Song Kang-ho", "Lee Sun-kyun", "Cho Yeo-jeong", "Choi Woo-shik", "Park So-dam", "Lee Jung-eun", "Jang Hye-jin", "Park Myung-hoon", "Jung Zi-so", "Jung Hyeon-jun"],
    76341: ["Tom Hardy", "Charlize Theron", "Nicholas Hoult", "Hugh Keays-Byrne", "Josh Helman", "Nathan Jones", "Zoë Kravitz", "Rosie Huntington-Whiteley", "Riley Keough", "Abbey Lee"],
    313369: ["Ryan Gosling", "Emma Stone", "John Legend", "Rosemarie DeWitt", "Finn Wittrock", "J.K. Simmons", "Sonoya Mizuno", "Jessica Rothe", "Callie Hernandez", "Tom Everett Scott"],
    37799: ["Jesse Eisenberg", "Andrew Garfield", "Justin Timberlake", "Armie Hammer", "Max Minghella", "Josh Pence", "Rooney Mara", "Joseph Mazzello", "Rashida Jones", "John Getz"],
    346698: ["Margot Robbie", "Ryan Gosling", "America Ferrera", "Kate McKinnon", "Issa Rae", "Rhea Perlman", "Will Ferrell", "Hari Nef", "Emma Mackey", "Simu Liu"],
    872585: ["Cillian Murphy", "Emily Blunt", "Matt Damon", "Robert Downey Jr.", "Florence Pugh", "Josh Hartnett", "Casey Affleck", "Rami Malek", "Kenneth Branagh", "Benny Safdie"],
    545611: ["Michelle Yeoh", "Stephanie Hsu", "Ke Huy Quan", "James Hong", "Jamie Lee Curtis", "Tallie Medel", "Jenny Slate", "Harry Shum Jr.", "Biff Wiff", "Sunita Mani"],
    578: ["Roy Scheider", "Robert Shaw", "Richard Dreyfuss", "Lorraine Gary", "Murray Hamilton", "Carl Gottlieb", "Jeffrey Kramer", "Susan Backlinie", "Jonathan Filley", "Ted Grossman"],
    129: ["Rumi Hiiragi", "Miyu Irino", "Mari Natsuki", "Takashi Naito", "Yasuko Sawaguchi", "Tatsuya Gashuin", "Ryunosuke Kamiki", "Yumi Tamai", "Yo Oizumi", "Koba Hayashi"],
    278: ["Tim Robbins", "Morgan Freeman", "Bob Gunton", "William Sadler", "Clancy Brown", "Gil Bellows", "Mark Rolston", "James Whitmore", "Jeffrey DeMunn", "Larry Brandenburg"],
    438631: ["Timothée Chalamet", "Rebecca Ferguson", "Oscar Isaac", "Josh Brolin", "Stellan Skarsgård", "Dave Bautista", "Zendaya", "Charlotte Rampling", "Jason Momoa", "Javier Bardem"],
    376867: ["Trevante Rhodes", "André Holland", "Janelle Monáe", "Ashton Sanders", "Jharrel Jerome", "Alex R. Hibbert", "Jaden Piner", "Naomie Harris", "Mahershala Ali", "Shariff Earp"],
}

print(len(SAMPLE), "sample titles")

In [ ]:
def load_movies_with_credits() -> pd.DataFrame | None:
    path = MOVIES_WITH_CREDITS_PATH
    if not path.exists():
        print(
            f"WARNING: {path} is missing. Run fetch_credits + merge_credits, then re-run.\n"
            "Scoring will use the tmdb.org fallback lists (same billing `order` as the API)."
        )
        return None
    df = pd.read_parquet(path)
    print(f"Loaded {len(df):,} rows from {path}")
    if "id" in df.columns:
        df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")
    return df


def billed_names(movie: dict, movies: pd.DataFrame | None, n: int = 10) -> tuple[list[str], str]:
    mid = movie["id"]
    if movies is not None:
        hit = movies.loc[movies["id"] == mid]
        if hit.empty and "title" in movies.columns:
            hit = movies.loc[movies["title"].str.casefold() == movie["title"].casefold()]
        if not hit.empty:
            names = [m.get("name") for m in sort_cast(hit.iloc[0].get("cast")) if m.get("name")]
            if names:
                return names[:n], "parquet"
    fallback = TMDB_TOP_BILLED_FALLBACK.get(mid, [])
    return fallback[:n], "tmdb.org snapshot 2026-09-10"


movies = load_movies_with_credits()

for movie in SAMPLE:
    names, source = billed_names(movie, movies, n=10)
    print("=" * 72)
    print(f"{movie['title']} ({movie['year']})  id={movie['id']}  [{movie['category']}]  source={source}")
    print("known leads:", ", ".join(movie["known_leads"][:4]), "…" if len(movie["known_leads"]) > 4 else "")
    if not names:
        print("  (no cast)")
        continue
    for i, name in enumerate(names):
        flag = "  <-- lead" if name_matches(name, movie["known_leads"]) else ""
        print(f"  order={i:>2}  {name}{flag}")

## Manual ground truth (top 2–3 leads, from film knowledge)

These are **not** copied from TMDB. Encoded as `known_leads` in the cell above.

| # | Title | Year | Real top leads (memory) |
|---|-------|------|-------------------------|
| 1 | The Avengers | 2012 | Downey, Evans, Hemsworth, Johansson, Ruffalo, Renner (ensemble; poster often leads with RDJ / Evans) |
| 2 | The Dark Knight | 2008 | Christian Bale, Heath Ledger, Aaron Eckhart |
| 3 | Inception | 2010 | Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page (Hardy / Cotillard also principal) |
| 4 | Titanic | 1997 | Leonardo DiCaprio, Kate Winslet |
| 5 | Avatar | 2009 | Sam Worthington, Zoe Saldaña |
| 6 | The Godfather | 1972 | Al Pacino is the protagonist; Marlon Brando is the billed star. James Caan third. |
| 7 | Pulp Fiction | 1994 | Travolta, Jackson, Thurman, Willis ("AND Bruce Willis" on the one-sheet) |
| 8 | Get Out | 2017 | Daniel Kaluuya, Allison Williams |
| 9 | Parasite | 2019 | Song Kang-ho and the Kim family (Choi Woo-shik is the POV); Lee Sun-kyun heads the Park house |
| 10 | Mad Max: Fury Road | 2015 | Charlize Theron is the dramatic lead; Tom Hardy is billed first as Max |
| 11 | La La Land | 2016 | Ryan Gosling, Emma Stone |
| 12 | The Social Network | 2010 | Jesse Eisenberg, Andrew Garfield |
| 13 | Barbie | 2023 | Margot Robbie, Ryan Gosling |
| 14 | Oppenheimer | 2023 | Cillian Murphy; Blunt / Damon / Downey as principal support |
| 15 | Everything Everywhere All at Once | 2022 | Michelle Yeoh, Ke Huy Quan, Stephanie Hsu |
| 16 | Jaws | 1975 | Roy Scheider, Robert Shaw, Richard Dreyfuss |
| 17 | Spirited Away | 2001 | Japanese voice leads: Rumi Hiiragi (Chihiro), Miyu Irino (Haku) — not the English dub |
| 18 | The Shawshank Redemption | 1994 | Tim Robbins, Morgan Freeman |
| 19 | Dune | 2021 | Timothée Chalamet; Ferguson / Zendaya / Isaac as principal support |
| 20 | Moonlight | 2016 | Chiron (Hibbert / Sanders / Rhodes) and Kevin across three ages; Ali / Harris supporting |

A title **agrees** if TMDB `order == 0` **and** `order == 1` are both in that known-lead set (slots match leads, not that folk memory of "#1" equals billed #1).

In [ ]:
rows = []
for movie in SAMPLE:
    names, source = billed_names(movie, movies, n=10)
    order0 = names[0] if len(names) > 0 else None
    order1 = names[1] if len(names) > 1 else None
    slot0_ok = bool(order0) and name_matches(order0, movie["known_leads"])
    slot1_ok = bool(order1) and name_matches(order1, movie["known_leads"])
    agree = slot0_ok and slot1_ok
    alpha = looks_alphabetical(names)
    if not names:
        verdict = "MISSING"
    elif alpha:
        verdict = "ALPHABETICAL"
    elif agree:
        verdict = "MATCHES_BILLING"
    else:
        verdict = "NOISY"
    rows.append(
        {
            "title": movie["title"],
            "year": movie["year"],
            "category": movie["category"],
            "order_0": order0,
            "order_1": order1,
            "slot0_is_lead": slot0_ok,
            "slot1_is_lead": slot1_ok,
            "agree": agree,
            "alphabetical_top10": alpha,
            "verdict": verdict,
            "source": source,
        }
    )

scorecard = pd.DataFrame(rows)
n = len(scorecard)
n_agree = int(scorecard["agree"].sum())
n_alpha = int(scorecard["alphabetical_top10"].sum())
n_missing = int((scorecard["verdict"] == "MISSING").sum())
n_noisy = int((scorecard["verdict"] == "NOISY").sum())

print(scorecard[["title", "year", "order_0", "order_1", "verdict", "source"]].to_string(index=False))
print()
print(f"Leads in TMDB order 0–1 slots: {n_agree}/{n} ({n_agree / n:.0%})")
print(f"Alphabetical top-10:           {n_alpha}/{n}")
print(f"Missing / empty cast:          {n_missing}/{n}")
print(f"Noisy (slots are not leads):   {n_noisy}/{n}")
print()
print("By category — agreement rate:")
print(scorecard.groupby("category")["agree"].agg(["sum", "count"]).assign(rate=lambda d: d["sum"] / d["count"]))

## Scorecard (filled from TMDB `order`, checked against the ground-truth table)

Verified against live TMDB `/cast` pages (2026-09-10) and recomputeable from the parquet when present. `order` on the website is 1-based display of the same 0-based API field.

| # | Title | Year | TMDB order 0–1 | Verdict | Notes |
|---|-------|------|----------------|---------|-------|
| 1 | The Avengers | 2012 | Robert Downey Jr., Chris Evans | MATCHES_BILLING | Ensemble; slots are two real leads, not an extra |
| 2 | The Dark Knight | 2008 | Christian Bale, Heath Ledger | MATCHES_BILLING | Exact poster billing |
| 3 | Inception | 2010 | Leonardo DiCaprio, Joseph Gordon-Levitt | MATCHES_BILLING | Watanabe is billed 3rd (above Page/Hardy) — billing, not alpha |
| 4 | Titanic | 1997 | Leonardo DiCaprio, Kate Winslet | MATCHES_BILLING | Pre-2000 two-hander, clean |
| 5 | Avatar | 2009 | Sam Worthington, Zoe Saldaña | MATCHES_BILLING | |
| 6 | The Godfather | 1972 | Marlon Brando, Al Pacino | MATCHES_BILLING | Pacino is the protagonist; Brando is correctly first-billed |
| 7 | Pulp Fiction | 1994 | John Travolta, Samuel L. Jackson | MATCHES_BILLING | Willis is 4th, not buried / not alphabetical |
| 8 | Get Out | 2017 | Daniel Kaluuya, Allison Williams | MATCHES_BILLING | Mid-budget, small principal cast |
| 9 | Parasite | 2019 | Song Kang-ho, Lee Sun-kyun | MATCHES_BILLING | Korean name order preserved; POV son Choi Woo-shik is 4th |
| 10 | Mad Max: Fury Road | 2015 | Tom Hardy, Charlize Theron | MATCHES_BILLING | Theron is the dramatic lead; Hardy is correctly billed first |
| 11 | La La Land | 2016 | Ryan Gosling, Emma Stone | MATCHES_BILLING | |
| 12 | The Social Network | 2010 | Jesse Eisenberg, Andrew Garfield | MATCHES_BILLING | |
| 13 | Barbie | 2023 | Margot Robbie, Ryan Gosling | MATCHES_BILLING | Recent |
| 14 | Oppenheimer | 2023 | Cillian Murphy, Emily Blunt | MATCHES_BILLING | Large cast; first two are principals |
| 15 | Everything Everywhere All at Once | 2022 | Michelle Yeoh, Stephanie Hsu | MATCHES_BILLING | Quan (co-lead) is 3rd — still a lead in 0–2 |
| 16 | Jaws | 1975 | Roy Scheider, Robert Shaw | MATCHES_BILLING | 1975 three-hander; Dreyfuss 3rd |
| 17 | Spirited Away | 2001 | Rumi Hiiragi, Miyu Irino | MATCHES_BILLING | Japanese voice cast, not the English dub |
| 18 | The Shawshank Redemption | 1994 | Tim Robbins, Morgan Freeman | MATCHES_BILLING | |
| 19 | Dune | 2021 | Timothée Chalamet, Rebecca Ferguson | MATCHES_BILLING | Zendaya is marketed hard but billed 7th — marketing ≠ billing |
| 20 | Moonlight | 2016 | Trevante Rhodes, André Holland | MATCHES_BILLING | Adult Chiron/Kevin first; kid/teen Chirons later. Not alphabetical |

**Agreement: 20 / 20** (leads occupy TMDB `order` 0 and 1).

Alphabetical top-10: **0 / 20**. Missing cast: **0 / 20** on these titles (re-check if a row is absent from the parquet).

## Failure patterns

No cluster of *wrong* `order` showed up in this sample. What *did* show up is **billing ≠ folk ranking**, which is not a TMDB bug:

| Pattern | Where | What it means for Path A |
|---|---|---|
| First-billed ≠ protagonist | Godfather (Brando > Pacino), Fury Road (Hardy > Theron) | Inverse weight follows the one-sheet, not "who the camera loves." Acceptable for a *star-power* feature. |
| Ensemble / special billing | Avengers, Pulp Fiction (Willis 4th), EEAAO (Quan 3rd), Dune (Zendaya 7th) | Extra leads sit at order 2–7, not at 40. Path A still up-weights them vs extras. |
| Multi-actor role | Moonlight (three Chirons) | Adults billed first. Inverse weights the Oscar-campaign names, not Little. |
| Foreign / anime | Parasite, Spirited Away | Local billed names, not English-dub or Westernized alphabetical dumps. |
| Pre-2000 | Godfather, Jaws, Pulp Fiction, Shawshank, Titanic | Same quality as 2010s titles in this set. |
| Alphabetical / missing | — | **Not observed** on these 20. That failure mode, if it exists, is in the obscure long tail — rows we already drop when budget/revenue are 0. |

Caveat: this is not a random 20-row sample of the full dump. It *is* representative of the financially valid movies star power will use. If later EDA on a random financially valid slice finds alphabetical `order` on a meaningful chunk, flip the flag back to Path B.

## Conclusion

`cast_order` matched known leads in the `order` 0–1 slots on **20/20** films, with **0** alphabetical lists, including older, foreign, indie, and ensemble titles — reliable enough for Path A (full-cast inverse-`cast_order` weights).

```python
# config.py — set from this notebook
USE_WEIGHTED_STAR_POWER = True  # Path A; 20/20 lead-slot agreement, 0/20 alphabetical
```

When implementing `star_power.py`, read that flag only. Use `1 / (order + 1)` because TMDB `order` is 0-based. Do not re-decide the path inside the feature module.

In [ ]:
RECOMMENDED_USE_WEIGHTED_STAR_POWER = True  # Path A; see conclusion cell

print(f"agreement {n_agree}/{n} | alphabetical {n_alpha}/{n} | missing {n_missing}/{n}")
print("RECOMMENDED_USE_WEIGHTED_STAR_POWER =", RECOMMENDED_USE_WEIGHTED_STAR_POWER)
print("config.USE_WEIGHTED_STAR_POWER     =", USE_WEIGHTED_STAR_POWER)
if USE_WEIGHTED_STAR_POWER != RECOMMENDED_USE_WEIGHTED_STAR_POWER:
    print("NOTE: config.py does not yet match this notebook — update USE_WEIGHTED_STAR_POWER.")
else:
    print("config.py already matches this notebook (Path A).")